# DSA 210 — ML Phase: Day 1
## Feature Engineering & Leakage Cleanup

**Bedirhan Ceylan** — Spring 2026

This notebook prepares features for predicting task delays. Day 1 focus: build leakage-free feature matrix from the EDA dataset.

## 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sollen_tasks.csv', parse_dates=['creation_date', 'completion_date'])
df = df.sort_values('creation_date').reset_index(drop=True)
print(f"Loaded: {df.shape[0]} tasks, {df.shape[1]} columns")
df.head(3)

Loaded: 486 tasks, 10 columns


,task_id,title,task_category,priority_level,assigned_to,creation_date,completion_date,actual_duration_days,delay_label,has_urgent_tag
0,2,Flutter initiate,tech,medium,Developer_A,2024-10-18 10:23:35,2024-11-20 17:29:55,33.296065,on_time,0
1,3,Login Screen - 1,tech,medium,Developer_B,2024-10-18 10:29:40,2024-11-20 17:29:49,33.291771,on_time,0
2,4,Backend login API,tech,medium,Developer_A,2024-10-18 10:29:51,2024-10-31 18:43:54,13.343090,on_time,0


## 2. Define Target & Identify Drops

**Target:** `is_delayed` (binary) — derived from existing `delay_label`.

**Dropped columns (and why):**
- `actual_duration_days`, `completion_date`, `delay_label` — **target leakage** (target was constructed from these)
- `priority_level` — 99.2% one value, no variance
- `has_urgent_tag` — 99.4% zero, no variance
- `task_id` — identifier, not a feature

In [2]:
y = (df['delay_label'] == 'delayed').astype(int)
print(f"Target — delayed: {y.sum()} ({y.mean()*100:.1f}%) | on_time: {(1-y).sum()} ({(1-y).mean()*100:.1f}%)")

leakage_cols = ['actual_duration_days', 'completion_date', 'delay_label']
dead_cols = ['priority_level', 'has_urgent_tag']
id_cols = ['task_id']

print(f"\nDropped — leakage: {leakage_cols}")
print(f"Dropped — no variance: {dead_cols}")
print(f"Dropped — identifier: {id_cols}")

Target — delayed: 169 (34.8%) | on_time: 317 (65.2%)

Dropped — leakage: ['actual_duration_days', 'completion_date', 'delay_label']
Dropped — no variance: ['priority_level', 'has_urgent_tag']
Dropped — identifier: ['task_id']


## 3. Base Features

Categorical and temporal features extracted directly from the source columns.

In [3]:
features = pd.DataFrame()

features['task_category'] = df['task_category']
features['assigned_to'] = df['assigned_to']
features['creation_day'] = df['creation_date'].dt.day_name()
features['creation_month_num'] = df['creation_date'].dt.month
features['is_weekend_creation'] = df['creation_date'].dt.dayofweek.isin([5, 6]).astype(int)
features['title_word_count'] = df['title'].str.split().str.len()

print(f"Shape so far: {features.shape}")
features.head()

Shape so far: (486, 6)


,task_category,assigned_to,creation_day,creation_month_num,is_weekend_creation,title_word_count
0,tech,Developer_A,Friday,10,0,2
1,tech,Developer_B,Friday,10,0,4
2,tech,Developer_A,Friday,10,0,3
3,tech,Developer_A,Friday,10,0,4
4,product,Developer_C,Friday,10,0,6


## 4. Workload Features

For each task, compute the system load **at the moment the task was created**:
- `team_total_load_at_creation` — total open tasks across the team
- `developer_workload_at_creation` — open tasks already assigned to the same developer

A task counts as "open at time T" if: `creation_date < T` AND `completion_date >= T`.

In [4]:
def open_tasks_at(creation_dt, df_full):
    mask = (df_full['creation_date'] < creation_dt) & (df_full['completion_date'] >= creation_dt)
    return int(mask.sum())

def open_tasks_for_dev(creation_dt, dev, df_full):
    mask = ((df_full['creation_date'] < creation_dt) &
            (df_full['completion_date'] >= creation_dt) &
            (df_full['assigned_to'] == dev))
    return int(mask.sum())

features['team_total_load_at_creation'] = df.apply(
    lambda r: open_tasks_at(r['creation_date'], df), axis=1)
features['developer_workload_at_creation'] = df.apply(
    lambda r: open_tasks_for_dev(r['creation_date'], r['assigned_to'], df), axis=1)

print(features[['team_total_load_at_creation', 'developer_workload_at_creation']].describe().round(2))

       team_total_load_at_creation  developer_workload_at_creation
count                       486.00                          486.00
mean                         32.30                            4.06
std                          13.72                            4.74
min                           0.00                            0.00
25%                          21.00                            1.00
50%                          30.00                            3.00
75%                          45.00                            5.00
max                          60.00                           30.00


## 5. Developer Historical Performance (Leakage-Free)

For each task, compute the developer's average performance **using only tasks that were created strictly before the current task**. This is critical — using all tasks would leak future information.

Tasks with no prior history (developer's first task) are filled with the global mean.

In [5]:
dev_past_delay = []
dev_past_duration = []

for _, row in df.iterrows():
    past = df[(df['assigned_to'] == row['assigned_to']) & (df['creation_date'] < row['creation_date'])]
    if len(past) == 0:
        dev_past_delay.append(np.nan)
        dev_past_duration.append(np.nan)
    else:
        dev_past_delay.append((past['delay_label'] == 'delayed').mean())
        dev_past_duration.append(past['actual_duration_days'].mean())

features['dev_historical_delay_rate'] = dev_past_delay
features['dev_avg_past_duration'] = dev_past_duration

# Cold-start: fill first-task-of-developer rows with global mean
features['dev_historical_delay_rate'] = features['dev_historical_delay_rate'].fillna(
    features['dev_historical_delay_rate'].mean())
features['dev_avg_past_duration'] = features['dev_avg_past_duration'].fillna(
    features['dev_avg_past_duration'].mean())

print(features[['dev_historical_delay_rate', 'dev_avg_past_duration']].describe().round(3))
print(f"\nMissing values across all features: {features.isnull().sum().sum()} (must be 0)")

       dev_historical_delay_rate  dev_avg_past_duration
count                    486.000                486.000
mean                       0.411                 40.217
std                        0.194                 27.558
min                        0.000                  0.001
25%                        0.250                 20.023
50%                        0.410                 36.643
75%                        0.533                 51.045
max                        1.000                295.020

Missing values across all features: 0 (must be 0)


## 6. Save Feature Matrix

Output: `data/sollen_tasks_features.csv` with all engineered features plus the binary target. Day 2 (modeling) will load this file directly.

In [6]:
output = features.copy()
output['is_delayed'] = y.values

output_path = '../data/sollen_tasks_features.csv'
output.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {output.shape}")
print(f"Columns ({len(output.columns)}):")
for c in output.columns:
    print(f"  - {c}")
print()
output.head()

Saved: ../data/sollen_tasks_features.csv
Shape: (486, 11)
Columns (11):
  - task_category
  - assigned_to
  - creation_day
  - creation_month_num
  - is_weekend_creation
  - title_word_count
  - team_total_load_at_creation
  - developer_workload_at_creation
  - dev_historical_delay_rate
  - dev_avg_past_duration
  - is_delayed



,task_category,assigned_to,creation_day,creation_month_num,is_weekend_creation,title_word_count,team_total_load_at_creation,developer_workload_at_creation,dev_historical_delay_rate,dev_avg_past_duration,is_delayed
0,tech,Developer_A,Friday,10,0,2,0,0,0.410902,40.216871,0
1,tech,Developer_B,Friday,10,0,4,1,0,0.410902,40.216871,0
2,tech,Developer_A,Friday,10,0,3,2,1,0.000000,33.296065,0
3,tech,Developer_A,Friday,10,0,4,3,2,0.000000,23.319578,0
4,product,Developer_C,Friday,10,0,6,4,0,0.410902,40.216871,0


## Day 1 Summary

- Loaded 486 task records, sorted chronologically
- Defined binary target `is_delayed` (34.8% positive class)
- Dropped 3 leakage columns + 2 zero-variance columns + 1 identifier
- Built 10 features: 4 base + 2 temporal + 2 workload + 2 historical
- Verified: 0 missing values
- Saved feature matrix to `data/sollen_tasks_features.csv`

**Next (Day 2):** Baseline classification models — Logistic Regression, Decision Tree, kNN — with stratified 5-fold cross-validation.